In [ ]:
using CoulombIntegral

In [ ]:
using SpecialFunctions
using FunctionZeros
using QuadGK
using LinearAlgebra
using Plots

In [ ]:
sph_besselj(l,z) = z == 0 ? (l==0 ? 1.0 : 0.0) : besselj(l+1/2, z) * sqrt(π/2z);
function wfn(nl, R)
    n,l = nl;
    k = besselj_zero(l+1/2, n) / R;
    norm = sqrt(2/R^3) * sph_besselj(l+1, k*R)^(-1) * (-1)^(n-1);
    return ( (f = x -> norm * sph_besselj(l, k*x), k = k) ); # NamedTuple
end

In [ ]:
function get_wfn!(nl, dict, R)
    haskey(dict, nl) && return dict[nl].f;
    wf = wfn(nl, R);
    dict[nl] = wf;
    return wf.f;
end

function get_kvec!(nl, dict, R)
    haskey(dict, nl) && return dict[nl].k;
    wf = wfn(nl, R);
    dict[nl] = wf;
    return wf.k;
end

In [ ]:
R1 = 1.0;
rwfns = Dict{Tuple,NamedTuple}() # (n,l) => (f=fun(x), k=k)

In [ ]:
x = (0:0.01:R1);
p = plot(; xlabel="r/R []", ylabel="function value [a.u.]", title="Radial wave functions");
for n in 1:3
    for l in 0:2
        y = get_wfn!((n,l), rwfns, R1).(x);
        plot!(p,x,y; label="n=$n,l=$l");
    end
end
display(p)

In [ ]:
for n in 1:3
    for l in 0:2
        int = quadgk(x->( x^2*get_wfn!( (n,l), rwfns, R1)(x)^2 ), 0, R1)[1];
        println("n: $n, l: $l, product: $int");
    end
end

In [ ]:
basis0 = [((1,0,0),(1,0,0)),       # |ss>
            ((1,1,-1),(1,1,-1)),   # |yy>
            ((1,1,0),(1,1,0)),     # |zz>
            ((1,1,1),(1,1,1))]     # |xx>

In [ ]:
b1 = [((1,0,0),(1,1,-1)),   # |sy>
      ((1,1,-1),(1,0,0)),   # |ys>
      ((1,0,0),(1,1,0)),    # |sz>
      ((1,1,0),(1,0,0)),    # |zs>
      ((1,0,0),(1,1,1)),    # |sx>
      ((1,1,1),(1,0,0))]    # |xs>

In [ ]:
coul0 = [coulomb_integral(Expand(), (nl)->get_wfn!(nl, rwfns, R1),fs...,is...; SH_basis=:real, recalc=true).int
    for fs in basis0, is in basis0]

In [ ]:
c1 = [coulomb_integral(Expand(), (nl)->get_wfn!(nl, rwfns, R1),fs...,is...; SH_basis=:real, recalc=true).int
    for fs in b1, is in b1]

In [ ]:
basis1 = [  ((1,0,0),(1,1,1)),    # |sx>
            ((1,1,1),(1,0,0))]    # |xs>

In [ ]:
coul1 = [coulomb_integral(Expand(), (nl)->get_wfn!(nl, rwfns, R1),fs...,is...; SH_basis=:real, recalc=true).int
    for fs in basis1, is in basis1]

In [ ]:
# <x|x|s> dipole moment
ap = 1/sqrt(3);
rp = quadgk(x->(x^3 * get_wfn!( (1,0), rwfns, R1)(x) * get_wfn!( (1,1), rwfns, R1)(x)), 0, R1)[1]
dip(A) = A*ap*rp;

In [ ]:
const hbar = 6.58212e-4; # eV ps
const ec = 1; # e
const perm0 = 55.26; # e V^(-1) um^(-1)
const gaas = 12.85 * perm0;
const m0 = 5.68; # eV ps^2 um^(-2)
const gamma = 1/(2700e-3); # ps^(-1) ... 2700 fs

In [ ]:
kin0(me, mh, A) = [fs==is ? ((hbar*get_kvec!((fs[1]), rwfns, R1))^2/(2me*m0*A^2) + # eV
                             (hbar*get_kvec!((fs[2]), rwfns, R1))^2/(2mh*m0*A^2)) : 0 for fs in basis0, is in basis0];

In [ ]:
kin1(me, mh, A) = [fs==is ? ((hbar*get_kvec!((fs[1]), rwfns, R1))^2/(2me*m0*A^2) + # eV
                             (hbar*get_kvec!((fs[2]), rwfns, R1))^2/(2mh*m0*A^2)) : 0 for fs in basis1, is in basis1];

In [ ]:
kin0(0.07,0.5,30e-3)

In [ ]:
kin1(0.07,0.5,30e-3)

In [ ]:
ham0(me, mh, A) = (kin0(me, mh, A) - ec*coul0/(4pi*gaas*A)); # eV
ham1(me, mh, A) = (kin1(me, mh, A) - ec*coul1/(4pi*gaas*A)); # eV

In [ ]:
h0 = ham0(0.07, 0.5, 30e-3);
eval0,evec0 = LinearAlgebra.eigen(h0)

In [ ]:
h1 = ham1(0.07, 0.5, 30e-3);
eval1,evec1 = LinearAlgebra.eigen(h1)

In [ ]:
function mobility(f::Union{Real,Vector}, eval0::Vector, evec0::Matrix, eval1::Vector, evec1::Matrix, dip::Real)
    w = 2pi.*f; # rad.ps^(-1)
    dip1 = dip*(evec0[1]-evec0[2])*(evec1[1]-evec1[2]); # um
    dip2 = dip*(evec0[1]-evec0[2])*(evec1[1]+evec1[2]); # um
    ediff1 = (eval1[1] - eval0[1]); # eV
    ediff2 = (eval1[2] - eval0[1]); # eV
    val =  1e4im .* w .* ec .* dip1^2 ./ (hbar.*w .- ediff1 .+ 1im*gamma*hbar); # cm^2.V^(-1).s^(-1)
    val += 1e4im .* w .* ec .* dip2^2 ./ (hbar.*w .- ediff2 .+ 1im*gamma*hbar); # cm^2.V^(-1).s^(-1)
    return val;
end

In [ ]:
function mobility_non(f::Union{Real,Vector}, eval0::Vector, evec0::Matrix, eval1::Vector, evec1::Matrix, dip::Real)
    w = 2pi.*f; # rad.ps^(-1)
    dip1 = dip; # um
    dip2 = dip; # um
    ediff1 = (eval1[1] - eval0[1]); # eV
    ediff2 = (eval1[2] - eval0[1]); # eV
    val =  1e4im .* w .* ec .* dip1^2 ./ (hbar.*w .- ediff1 .+ 1im*gamma*hbar); # cm^2.V^(-1).s^(-1)
    val += 1e4im .* w .* ec .* dip2^2 ./ (hbar.*w .- ediff2 .+ 1im*gamma*hbar); # cm^2.V^(-1).s^(-1)
    return val;
end

In [ ]:
mres = 60;
fres = 100;
data_c = zeros(Float64, mres, fres);
data_n = zeros(Float64, mres, fres);
fmax = 3;
freq = collect(0:fmax/(fres-1):fmax);

In [ ]:
mtot = 0.067 + 0.47;
A1 = 25e-3; # um (25 nm)

In [ ]:
for itr in 1:mres
    me = mtot * (0.9*(itr-1)/(mres-1) + 0.05);
    mh = mtot - me;
    h0 = ham0(me, mh, A1);
    eval0, evec0 = LinearAlgebra.eigen(h0);
    h1 = ham1(me, mh, A1);
    eval1, evec1 = LinearAlgebra.eigen(h1);
    data_c[itr,:] = mobility(freq, eval0, evec0, eval1, evec1, dip(A1)) |> real;
end

In [ ]:
for itr in 1:mres
    me = mtot * (0.9*(itr-1)/(mres-1) + 0.05);
    mh = mtot - me;
    h0 = kin0(me, mh, A1);
    eval0, evec0 = LinearAlgebra.eigen(h0);
    h1 = kin1(me, mh, A1);
    eval1, evec1 = LinearAlgebra.eigen(h1);
    data_n[itr,:] = mobility_non(freq, eval0, evec0, eval1, evec1, dip(A1)) |> real;
end

In [ ]:
m_vals = (0.05:0.9/(mres-1):0.95);
zero_log = zeros(mres,fres);
zero_log[:,1] += ones(mres,1).*0.1;

In [ ]:
heatmap(freq, m_vals, data_c.+zero_log; title="Coulomb interaction", clim=(100,5000),
                        xlabel="Frequency [THz]", ylabel="me/mtot []", colorbar_title="Re Mobility [a.u.]", colorbar_scale=:log10)

In [ ]:
heatmap(freq, m_vals, data_n.+zero_log; title="No interaction", clim=(100,5000),
                        xlabel="Frequency [THz]", ylabel="me/mtot []", colorbar_title="Re Mobility [a.u.]", colorbar_scale=:log10)